In [21]:
import serpapi
import json
import requests
import pandas as pd
import time

### One call

In [22]:
api_key = "e5700b2bd25f65569b59e1b9edc0718d208f3033d46b4eb69232ecc93f27f943"
api_key2 = "cae0d822c2848a3edc91a6fe613ab32ec3ad5f7e3769490a7f5e20c8b84a24ea"
api_key3 = "dc3e23721e31669446e540cc5458ac9a7b9eb9950dbfa04828f00190c66595cb"
api_key4 = "f7460fb648d4d97ba0a28ca2372209b84d48e84dbee044a3ef8b2d233b32fdbf" 

### Functions

In [23]:
def query_serpapi(location, search_query: str, api_key: str) -> dict:
    client = serpapi.Client(api_key=api_key)

    results = client.search({
        "q": search_query,
        "location": location,
        "hl": "en",
        "gl": "ng",
        "google_domain": "google.com.ng"
    })
    
    return results

In [24]:
def resolve_refs(indexes: list, ref_lookup: dict) -> list:
    """Map reference_indexes back to actual source metadata."""
    return [
        ref_lookup[i]
        for i in indexes
        if i in ref_lookup
    ]
def parse_ai_overview(ai_overview: dict) -> dict:

    ref_lookup = {
        ref["index"]: {
            "title": ref.get("title", ""),
            "link": ref.get("link", ""),
            "source": ref.get("source", "")
        }
        for ref in ai_overview.get("references", [])
    }

    parsed_sections = []

    for block in ai_overview.get("text_blocks", []):
        block_type = block.get("type")

        if block_type == "paragraph":
            parsed_sections.append({
                "type": "paragraph",
                "content": block.get("snippet", ""),
                "cited_sources": resolve_refs(block.get("reference_indexes", []), ref_lookup)
            })

        elif block_type == "list":
            items = [item.get("snippet", "") for item in block.get("list", [])]
            parsed_sections.append({
                "type": "list",
                "items": items,
                "cited_sources": resolve_refs(block.get("reference_indexes", []), ref_lookup)
            })

    return {
        "sections": parsed_sections,
        "all_references": list(ref_lookup.values())
    }

In [25]:
def extract_paa(serpapi_response: dict) -> list:
    """Extract People Also Ask questions — great seed for new queries."""
    return [
        item.get("question", "")
        for item in serpapi_response.get("related_questions", [])
    ]

### Loop through queries

In [26]:
queries = [
    "breast cancer",
    "what is cancer",
    "cancer symptoms",
    "lung cancer",
    "skin cancer",
    "colon cancer",
    "cancer treatment",
    "prostate cancer",
    "blood cancer",
    "pancreatic cancer",
    
    "diabetes type 2",
    "type 2 diabetes",
    "what is diabetes",
    "diabetes type 1",
    "type 1 diabetes",
    "diabetes symptoms",
    "gestational diabetes",
    "insulin",
    "symptoms of diabetes",
    "diabetes test",
    
    "hypertension blood pressure",
    "blood pressure",
    "pulmonary hypertension",
    "what is hypertension",
    "hypertension symptoms",
    "high blood pressure",
    "hypertension treatment",
    "intracranial hypertension",
    "stage 2 hypertension",
    "hypertension stage 2",
    
    "anxiety",
    "what is depression",
    "anxiety and depression",
    "depression and anxiety",
    "depression treatment",
    "depression symptoms",
    "depression help",
    "postpartum",
    "postpartum depression",
    "depression medication",
    
    "the hantavirus",
    "virus",
    "hantavirus symptoms",
    "virus hantavirus",
    "hantavirus outbreak",
    "virus hantavirus outbreak",
    "what is hantavirus",
    "hantavirus 2026",
    "hantavirus cruise",
    "hantavirus update",
    
    "covid symptoms",
    "long covid symptoms",
    "what is long covid",
    "how long does covid last",
    "long term covid",
    "how long was covid",
    "symptoms of covid",
    "symptoms of long covid",
    "long covid treatment",
    "covid 19",
    
    "acupuncture near me",
    "what is acupuncture",
    "acupuncture points",
    "acupuncture for pain",
    "acupuncture",
    "chinese acupuncture",
    "community acupuncture",
    "does acupuncture work",
    "fertility acupuncture",
    "acupuncture needles",
    
    "cupping therapy near me",
    "cupping near me",
    "what is cupping therapy",
    "cupping massage therapy",
    "massage cupping therapy",
    "cupping massage",
    "what is cupping",
    "massage",
    "cupping therapy benefits",
    "cupping set",
    
    "herbal chinese medicine",
    "chinese medicine",
    "chinese herbal medicine",
    "what is herbal medicine",
    "traditional herbal medicine",
    "herbal medicine near me",
    "acupuncture",
    "acupuncture herbal medicine",
    "herbal medicine school",
    "herbal tea",
    
    "what is abortion",
    "abortion clinic",
    "abortion laws",
    "planned parenthood",
    "planned parenthood abortion",
    "medical abortion",
    "what is an abortion",
    "supreme court abortion",
    "abortion pill online",
    "abortion meaning",
    
    "the hiv",
    "what hiv",
    "what is hiv",
    "aids",
    "hiv aids",
    "hiv symptoms",
    "hiv test",
    "hiv positive",
    "hiv treatment",
    "hiv testing",
    
    "gender affirming care",
    "gender affirming surgery",
    "what is gender affirming care",
    "gender affirming care definition",
    "gender affirming care near me",
    "gender affirming procedure",
    "gender reassignment surgery",
    "gender reassignment surgery male to female",
    "gender affirmation surgery",
    "how does gender reassignment surgery work"
]

In [27]:
len(queries)

120

In [28]:
def run_pipeline_for_queries(queries, api_key, locations):
    rows = []
    total = len(queries) * len(locations)
    counter = 0

    for location in locations:
        for i, query in enumerate(queries):
            counter += 1
            print(f"[{counter}/{total}] Location: '{location}' | Query: '{query}'")
            
            try:
                # 1. Hit SerpAPI
                results = query_serpapi(location, query, api_key)
                
                # 2. Extract AI Overview
                ai_overview_raw = results.get("ai_overview")
                
                # 3. Extract PAA
                paa = extract_paa(results)
                
                if not ai_overview_raw:
                    rows.append({
                        "location": location,
                        "query": query,
                        "has_ai_overview": False,
                        "all_items": None,
                        "cited_sources": None,
                        "all_references": None,
                        "paa_questions": paa
                        
                    })
                    continue

                # 4. Parse AI Overview
                parsed = parse_ai_overview(ai_overview_raw)

                # 5. Flatten all sections into one row
                all_items = []
                all_sources = []

                for section in parsed["sections"]:
                    if section["type"] == "list":
                        all_items.extend(section.get("items", []))
                    all_sources.extend(section.get("cited_sources", []))

                # Deduplicate sources
                seen = set()
                unique_sources = []
                for s in all_sources:
                    if s["link"] not in seen:
                        seen.add(s["link"])
                        unique_sources.append(s)

                rows.append({
                    "location": location,
                    "query": query,
                    "has_ai_overview": True,
                    "all_items": all_items,
                    "cited_sources": unique_sources,
                    "all_references": parsed["all_references"],
                    "paa_questions": paa
                })

            except Exception as e:
                print(f"   ERROR on '{query}' @ '{location}': {e}")
                rows.append({
                    "location": location,
                    "query": query,
                    "has_ai_overview": False,
                    "all_items": None,
                    "cited_sources": None,
                    "all_references": None,
                    "paa_questions": None
                })

            time.sleep(1.5)

    return pd.DataFrame(rows)

In [ ]:
df = run_pipeline_for_queries(queries, api_key4, locations = ['Nigeria'])

print(f"\nTotal rows: {len(df)}")
print(f"Queries with AI Overview: {df['has_ai_overview'].sum()}")
print(f"Queries without AI Overview: {(~df['has_ai_overview']).sum()}")
df.head()

[1/120] Location: 'Nigeria' | Query: 'breast cancer'
[2/120] Location: 'Nigeria' | Query: 'what is cancer'
[3/120] Location: 'Nigeria' | Query: 'cancer symptoms'
[4/120] Location: 'Nigeria' | Query: 'lung cancer'
[5/120] Location: 'Nigeria' | Query: 'skin cancer'
[6/120] Location: 'Nigeria' | Query: 'colon cancer'
[7/120] Location: 'Nigeria' | Query: 'cancer treatment'
[8/120] Location: 'Nigeria' | Query: 'prostate cancer'
[9/120] Location: 'Nigeria' | Query: 'blood cancer'
[10/120] Location: 'Nigeria' | Query: 'pancreatic cancer'
[11/120] Location: 'Nigeria' | Query: 'diabetes type 2'
[12/120] Location: 'Nigeria' | Query: 'type 2 diabetes'
   ERROR on 'type 2 diabetes' @ 'Nigeria': 522 Server Error: <none> for url: https://serpapi.com/search?q=type+2+diabetes&location=Nigeria&hl=en&gl=ng&google_domain=google.com.ng&api_key=f7460fb648d4d97ba0a28ca2372209b84d48e84dbee044a3ef8b2d233b32fdbf
[13/120] Location: 'Nigeria' | Query: 'what is diabetes'
[14/120] Location: 'Nigeria' | Query: 'dia

In [20]:
rows

NameError: name 'rows' is not defined

In [ ]:
file_name = "nigeria.csv"

df.to_csv(file_name, index=False) 
print("Saved to", file_name) 